In [2]:
from nilearn import *
import matplotlib as mpl
import pandas as pd


In [4]:
dataset = datasets.fetch_haxby()

[fetch_haxby] Dataset directory found: /Users/sonakshisharma/nilearn_data/haxby2001


In [5]:
labels = pd.read_csv(dataset.session_target[0], sep=" ")


In [6]:
conditions = labels["labels"] 
runs = labels["chunks"]

In [7]:
img = image.load_img(dataset.func[0])
print(f"Functional nifti image (4D) is of shape: {img.shape}")


Functional nifti image (4D) is of shape: (40, 64, 64, 1452)


In [8]:
func_filename = dataset.func[0]

In [9]:
from nilearn.maskers import NiftiMasker

masker = NiftiMasker(
    mask_img=dataset.mask_vt[0],
    smoothing_fwhm=8,
    standardize="zscore_sample",
    memory="nilearn_cache",
    memory_level=1,
    mask_strategy="epi",
    verbose=1,
)
data_masked = masker.fit_transform(func_filename)

[NiftiMasker.wrapped] Loading data from .../subj2/bold.nii.gz
[NiftiMasker.wrapped] Loading mask from .../mask4_vt.nii.gz
[NiftiMasker.wrapped] Resampling mask
[NiftiMasker.wrapped] Finished fit


In [10]:
conditions.value_counts()

labels
rest            588
scissors        108
face            108
cat             108
shoe            108
house           108
scrambledpix    108
bottle          108
chair           108
Name: count, dtype: int64

In [11]:
keep = conditions.isin(["face", "house"]).values
X_fh = data_masked[keep]
y_fh = conditions[keep].values
groups_fh = runs[keep].values

In [ ]:
print(data_masked.shape, X_fh.shape)  
print(pd.Series(y_fh).value_counts())  
print(pd.Series(groups_fh).unique())

(1452, 464) (216, 464)
face     108
house    108
Name: count, dtype: int64
[ 0  1  2  3  4  5  6  7  8  9 10 11]


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
train = groups_fh < 10
test = groups_fh >= 10

X_train, y_train = X_fh[train], y_fh[train]
X_test, y_test = X_fh[test], y_fh[test]

In [15]:
clf = LogisticRegression(max_iter=10000, random_state=0)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test)) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 100.00%


In [16]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.svm import SVC

cv = LeaveOneGroupOut()

for name, model in [
    ("Linear SVM", SVC(kernel="linear")),
    ("Logistic regression", LogisticRegression(max_iter=1000, random_state=0)),
]:
    scores = cross_val_score(model, X_fh, y_fh, groups=groups_fh, cv=cv)
    print(f"{name}: {scores.mean():.3f} (std {scores.std():.3f})")

Linear SVM: 0.986 (std 0.033)
Logistic regression: 0.981 (std 0.041)


In [17]:
from sklearn.dummy import DummyClassifier

dummy_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    X_fh, y_fh, groups=groups_fh, cv=cv,
)
print(f"Dummy accuracy: {dummy_scores.mean():.3f}")

Dummy accuracy: 0.500
